<a href="https://colab.research.google.com/github/MilagrosAlonso/TP-IA/blob/main/agente_ventas_mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Agente de Análisis de Ventas
### LangChain + Pandas + Mistral Small (sin RAG)
Dataset: `sales_data_normalized.csv`

## 1. Instalación de dependencias

In [24]:
!pip install -q -U langchain langchain-mistralai langchain-experimental pandas tabulate

## 2. Configuración de credenciales

In [25]:
import os

# Opción A: setear directamente (solo para pruebas locales)
# os.environ["MISTRAL_API_KEY"] = "tu_api_key_aqui"

# Opción B: Colab Secrets (recomendado en Google Colab)
# from google.colab import userdata
# os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

# Opción C: input manual
import getpass
if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Ingresá tu Mistral API Key: ")

print("✅ API Key configurada")

✅ API Key configurada


## 3. Carga del dataset normalizado

In [26]:
import pandas as pd

# Si estás en Colab, subí el archivo o usá la ruta correcta
from google.colab import files
uploaded = files.upload()  # subir sales_data_normalized.csv

# Modificamos el nombre del archivo para que coincida con el subido
df = pd.read_csv('sales_data_normalized (2).csv', parse_dates=["orderdate"])

print(f"✅ Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"📅 Período: {df['orderdate'].min().date()} → {df['orderdate'].max().date()}")
print(f"\nColumnas disponibles:")
print(df.dtypes)

Saving sales_data_normalized (2).csv to sales_data_normalized (2) (2).csv
✅ Dataset cargado: 2823 filas x 22 columnas
📅 Período: 2003-01-06 → 2005-05-31

Columnas disponibles:
ordernumber                  int64
quantityordered              int64
priceeach                  float64
orderlinenumber              int64
sales                      float64
orderdate           datetime64[ns]
status                      object
productline                 object
msrp                         int64
productcode                 object
customername                object
phone                       object
addressline1                object
addressline2                object
city                        object
state                       object
postalcode                  object
country                     object
territory                   object
contactlastname             object
contactfirstname            object
dealsize                    object
dtype: object


In [27]:
# Vista previa
df.head(3)

,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,productline,msrp,productcode,...,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,Motorcycles,95,S10_1678,...,897 Long Airport Avenue,0,NYC,NY,10022,USA,0,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,Motorcycles,95,S10_1678,...,59 rue de l'Abbaye,0,Reims,0,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,Motorcycles,95,S10_1678,...,27 rue du Colonel Pierre Avia,0,Paris,0,75508,France,EMEA,Da Cunha,Daniel,Medium


## 4. Inicialización del LLM (Mistral Small)

In [32]:
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0,          # 0 = más determinista, ideal para análisis de datos
    max_retries=3,
)

print(f"✅ LLM listo: {llm.model}")

✅ LLM listo: mistral-small-latest


## 5. Creación del Agente Pandas

In [36]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    verbose=True,
    allow_dangerous_code=True,
    agent_type="tool-calling",
    prefix="""Eres un analista de ventas experto. Tienes acceso a un dataset de ventas
con 2823 registros. El dataframe se llama `df` y contiene las columnas:
ordernumber, quantityordered, priceeach, orderlinenumber, sales, orderdate,
status, productline, msrp, productcode, customername, phone, addressline1,
addressline2, city, state, postalcode, country, territory, contactlastname,
contactfirstname, dealsize.

Responde siempre en español, con claridad y precisión.
Cuando sea útil, formatea los números con separadores de miles.""",
)

print("✅ Agente creado y listo para consultas")

✅ Agente creado y listo para consultas


## 6. Función auxiliar para consultas

In [38]:
def preguntar(consulta: str):
    """Envía una consulta al agente y muestra la respuesta."""
    print(f"\n{'='*60}")
    print(f"🧑 Pregunta: {consulta}")
    print(f"{'='*60}")
    respuesta = agent.invoke(consulta)
    print(f"\n🤖 Respuesta final:")
    print(respuesta["output"])
    print(f"{'='*60}\n")
    return respuesta["output"]

## 7. Ejemplos de consultas al agente

A continuación se muestran consultas de ejemplo. Podés modificarlas o agregar las tuyas.

In [39]:
# Consulta 1: Resumen general
preguntar("¿Cuáles son las ventas totales y el número de órdenes en el dataset?")


🧑 Pregunta: ¿Cuáles son las ventas totales y el número de órdenes en el dataset?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "ventas_totales = df['sales'].sum()\nnumero_ordenes = df['ordernumber'].nunique()\n\nventas_totales, numero_ordenes"}`


(np.float64(10032628.85), 307)Las ventas totales en el dataset son **10.032.628,85** y el número total de órdenes únicas es **307**.

> Finished chain.

🤖 Respuesta final:
Las ventas totales en el dataset son **10.032.628,85** y el número total de órdenes únicas es **307**.



'Las ventas totales en el dataset son **10.032.628,85** y el número total de órdenes únicas es **307**.'

In [40]:
# Consulta 2: Ranking de líneas de productos
preguntar("¿Cuál es la línea de productos más vendida por total de ventas? Mostrá el ranking completo.")


🧑 Pregunta: ¿Cuál es la línea de productos más vendida por total de ventas? Mostrá el ranking completo.


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "ranking_productline = df.groupby('productline')['sales'].sum().sort_values(ascending=False)\nranking_productline"}`


productline
Classic Cars        3919615.66
Vintage Cars        1903150.84
Motorcycles         1166388.34
Trucks and Buses    1127789.84
Planes               975003.57
Ships                714437.13
Trains               226243.47
Name: sales, dtype: float64El ranking completo de las líneas de productos más vendidas por **total de ventas** es el siguiente:

| **Posición** | **Línea de Producto** | **Total de Ventas (USD)** |
|--------------|-----------------------|---------------------------|
| 1            | Classic Cars          | 3.919.615,66              |
| 2            | Vintage Cars          | 1.903.150,84              |
| 3            | Motorcycles           | 1.166.388,34    

'El ranking completo de las líneas de productos más vendidas por **total de ventas** es el siguiente:\n\n| **Posición** | **Línea de Producto** | **Total de Ventas (USD)** |\n|--------------|-----------------------|---------------------------|\n| 1            | Classic Cars          | 3.919.615,66              |\n| 2            | Vintage Cars          | 1.903.150,84              |\n| 3            | Motorcycles           | 1.166.388,34              |\n| 4            | Trucks and Buses      | 1.127.789,84              |\n| 5            | Planes                | 975.003,57                |\n| 6            | Ships                 | 714.437,13                |\n| 7            | Trains                | 226.243,47                |\n\n**Conclusión:** La línea de productos **Classic Cars** es la más vendida, con un total de **3.919.615,66 USD**, seguida por **Vintage Cars** con **1.903.150,84 USD**.'

In [41]:
# Consulta 3: Análisis por país
preguntar("¿Cuáles son los 5 países con mayor volumen de ventas totales?")


🧑 Pregunta: ¿Cuáles son los 5 países con mayor volumen de ventas totales?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "ventas_por_pais = df.groupby('country')['sales'].sum().sort_values(ascending=False)\nventas_por_pais.head(5)"}`


country
USA          3627982.83
Spain        1215686.92
France       1110916.52
Australia     630623.10
UK            478880.46
Name: sales, dtype: float64Los **5 países con mayor volumen de ventas totales** son:

1. **USA**: **3.627.982,83**
2. **Spain**: **1.215.686,92**
3. **France**: **1.110.916,52**
4. **Australia**: **630.623,10**
5. **UK**: **478.880,46**

> Finished chain.

🤖 Respuesta final:
Los **5 países con mayor volumen de ventas totales** son:

1. **USA**: **3.627.982,83**
2. **Spain**: **1.215.686,92**
3. **France**: **1.110.916,52**
4. **Australia**: **630.623,10**
5. **UK**: **478.880,46**



'Los **5 países con mayor volumen de ventas totales** son:\n\n1. **USA**: **3.627.982,83**\n2. **Spain**: **1.215.686,92**\n3. **France**: **1.110.916,52**\n4. **Australia**: **630.623,10**\n5. **UK**: **478.880,46**'

In [42]:
# Consulta 4: Tendencia temporal
preguntar("¿Cómo evolucionaron las ventas por año? ¿Hubo crecimiento?")


🧑 Pregunta: ¿Cómo evolucionaron las ventas por año? ¿Hubo crecimiento?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Asegurarnos de que la columna 'orderdate' sea de tipo datetime\ndf['orderdate'] = pd.to_datetime(df['orderdate'])\n\n# Extraer el año de la fecha\ndf['year'] = df['orderdate'].dt.year\n\n# Agrupar por año y sumar las ventas\nventas_por_año = df.groupby('year')['sales'].sum().reset_index()\n\n# Ordenar por año\nventas_por_año = ventas_por_año.sort_values('year')\n\nventas_por_año"}`


   year       sales
0  2003  3516979.54
1  2004  4724162.60
2  2005  1791486.71
Invoking: `python_repl_ast` with `{'query': "# Calcular la variación porcentual anual\nventas_por_año['variacion_porcentual'] = ventas_por_año['sales'].pct_change() * 100\n\n# Redondear para mejor legibilidad\nventas_por_año['variacion_porcentual'] = ventas_por_año['variacion_porcentual'].round(2)\n\nventas_por_año"}`


   year       sales  variaci

'### Evolución de las ventas por año\n\nA continuación se presenta la evolución de las ventas anuales en el dataset, junto con la variación porcentual respecto al año anterior:\n\n| **Año** | **Ventas (USD)**       | **Variación porcentual** |\n|---------|------------------------|--------------------------|\n| 2003    | 3.516.979,54           | -                        |\n| 2004    | 4.724.162,60           | **+34,32%**              |\n| 2005    | 1.791.486,71           | **-62,08%**              |\n\n---\n\n### Análisis:\n1. **Crecimiento en 2004**:\n   - Las ventas aumentaron significativamente en un **34,32%** respecto a 2003, pasando de **3.516.979,54 USD** a **4.724.162,60 USD**.\n   - Esto indica un año de fuerte expansión en las ventas.\n\n2. **Caída en 2005**:\n   - En 2005, las ventas cayeron drásticamente un **62,08%**, reduciéndose a **1.791.486,71 USD**.\n   - Esta disminución sugiere un posible problema en el negocio, como una crisis de mercado, cambios en la estrategia o 

In [43]:
# Consulta 5: Análisis de deal size
preguntar("¿Qué porcentaje de las ventas corresponde a cada categoría de deal size (Small, Medium, Large)?")


🧑 Pregunta: ¿Qué porcentaje de las ventas corresponde a cada categoría de deal size (Small, Medium, Large)?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Verificar si 'sales' es numérico y calcular el total de ventas por 'dealsize'\ntotal_sales_by_dealsize = df.groupby('dealsize')['sales'].sum()\n\n# Calcular el total de ventas general\ntotal_sales = df['sales'].sum()\n\n# Calcular el porcentaje de ventas por categoría de 'dealsize'\npercentage_by_dealsize = (total_sales_by_dealsize / total_sales) * 100\n\npercentage_by_dealsize"}`


dealsize
Large     12.978844
Medium    60.676342
Small     26.344813
Name: sales, dtype: float64El porcentaje de las ventas correspondiente a cada categoría de **deal size** es el siguiente:

- **Large**: 12,98%
- **Medium**: 60,68%
- **Small**: 26,34%

Estos valores representan la distribución porcentual de las ventas totales según el tamaño del pedido. La categoría **Medium** es la que gen

'El porcentaje de las ventas correspondiente a cada categoría de **deal size** es el siguiente:\n\n- **Large**: 12,98%\n- **Medium**: 60,68%\n- **Small**: 26,34%\n\nEstos valores representan la distribución porcentual de las ventas totales según el tamaño del pedido. La categoría **Medium** es la que genera la mayor proporción de ventas, seguida por **Small** y luego **Large**.'

In [46]:
# Consulta 6: Clientes top
preguntar("¿Cuáles son los 10 clientes que más compraron en términos de ventas totales?")


🧑 Pregunta: ¿Cuáles son los 10 clientes que más compraron en términos de ventas totales?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "top_clientes = df.groupby('customername')['sales'].sum().nlargest(10)\ntop_clientes"}`


customername
Euro Shopping Channel           912294.11
Mini Gifts Distributors Ltd.    654858.06
Australian Collectors, Co.      200995.41
Muscle Machine Inc              197736.94
La Rochelle Gifts               180124.90
Dragon Souveniers, Ltd.         172989.68
Land of Toys Inc.               164069.44
The Sharp Gifts Warehouse       160010.27
AV Stores, Co.                  157807.81
Anna's Decorations, Ltd         153996.13
Name: sales, dtype: float64Los **10 clientes que más compraron en términos de ventas totales** son los siguientes:

| **Posición** | **Nombre del Cliente**               | **Ventas Totales (USD)** |
|--------------|--------------------------------------|--------------------------|
| 1            | Eur

"Los **10 clientes que más compraron en términos de ventas totales** son los siguientes:\n\n| **Posición** | **Nombre del Cliente**               | **Ventas Totales (USD)** |\n|--------------|--------------------------------------|--------------------------|\n| 1            | Euro Shopping Channel                | 912.294,11               |\n| 2            | Mini Gifts Distributors Ltd.         | 654.858,06               |\n| 3            | Australian Collectors, Co.           | 200.995,41               |\n| 4            | Muscle Machine Inc                   | 197.736,94               |\n| 5            | La Rochelle Gifts                    | 180.124,90               |\n| 6            | Dragon Souveniers, Ltd.              | 172.989,68               |\n| 7            | Land of Toys Inc.                    | 164.069,44               |\n| 8            | The Sharp Gifts Warehouse            | 160.010,27               |\n| 9            | AV Stores, Co.                       | 157.807,81  

In [ ]:
# Consulta 7: Estado de órdenes
preguntar("¿Cuántas órdenes hay en cada estado (Shipped, Cancelled, etc.) y qué porcentaje representan?")


🧑 Pregunta: ¿Cuántas órdenes hay en cada estado (Shipped, Cancelled, etc.) y qué porcentaje representan?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Contar el número de órdenes por estado\nstatus_counts = df['status'].value_counts()\n\n# Calcular el porcentaje que representa cada estado\ntotal_orders = len(df)\nstatus_percentages = (status_counts / total_orders) * 100\n\n# Crear un DataFrame con los resultados\nstatus_summary = pd.DataFrame({\n    'Cantidad de Órdenes': status_counts,\n    'Porcentaje (%)': status_percentages.round(2)\n})\n\nstatus_summary"}`


            Cantidad de Órdenes  Porcentaje (%)
status                                         
Shipped                    2617           92.70
Cancelled                    60            2.13
Resolved                     47            1.66
On Hold                      44            1.56
In Process                   41            1.45
Disputed                    

## 8. Chat interactivo

Escribí tus propias preguntas en bucle. Ingresá `salir` para terminar.

In [ ]:
print("💬 Modo interactivo — escribí tu pregunta (o 'salir' para terminar)\n")

while True:
    consulta = input("Vos: ").strip()
    if not consulta:
        continue
    if consulta.lower() in ["salir", "exit", "quit"]:
        print("👋 ¡Hasta luego!")
        break
    preguntar(consulta)

💬 Modo interactivo — escribí tu pregunta (o 'salir' para terminar)

Vos: Cual el el producto mas vendido?

🧑 Pregunta: Cual el el producto mas vendido?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "producto_mas_vendido = df.groupby('productcode')['quantityordered'].sum().idxmax()\nproducto_mas_vendido_info = df[df['productcode'] == producto_mas_vendido].iloc[0]\nproducto_mas_vendido_info"}`


ordernumber                         10104
quantityordered                        23
priceeach                           100.0
orderlinenumber                        13
sales                             4556.99
orderdate             2003-01-31 00:00:00
status                            Shipped
productline                  Classic Cars
msrp                                  169
productcode                      S18_3232
customername        Euro Shopping Channel
phone                      (91) 555 94 44
addressline1           C/ Moralzarzal, 86
addressline2      